SoundWall converter. This python script will grab data from the spreadsheets datavv.xlsx, datacc.xlsx, datavv2.xlsx and datacc2.xlsx and produce a json file that will construct the soundwall interactive.

import pandas as pd

def excel_to_json_string(input_file, output_file):
    # Read the Excel file
    excel_file = pd.ExcelFile(input_file)
    all_data = []

    # Iterate through each sheet in the Excel file
    for sheet_index, sheet_name in enumerate(excel_file.sheet_names):
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_suffix = f"{sheet_index + 1:02d}"  # Ensure unique IDs for each tab

        # Advanced Constructor
        if input_file in ['datacc2.xlsx', 'datavv2.xlsx']:
            vidPrefix = 'vv2'
            if input_file == 'datacc2.xlsx':
                vidPrefix = 'cc2'
                
            first_row = df.iloc[0]
            vlink = str(first_row.get("vlink", ""))
            all_data.append(
                "{\n"
                '  "type": "icn",\n'
                '  "role": "button",\n'
                '  "alt": "' + first_row.get("alt", "").replace('"', '\\"') + '",\n'
                '  "id": "icn' + sheet_suffix + '",\n'
                '  "content": "' + first_row.get("content", "").replace('"', '\\"') + '",\n'
                '  "left": "' + first_row.get("left", "").replace('"', '\\"') + '",\n'
                '  "top": "' + first_row.get("top", "").replace('"', '\\"') + '",\n'
                '  "vlink": "' + vlink + '",\n'
                '  "action": "openGroupGhost",\n'
                '  "target": "grp' + sheet_suffix + '"\n'
                '}'
            )

            group = "{\n" + \
                '  "type": "grp",\n' + \
                '  "id": "grp' + sheet_suffix + '",\n' + \
                '  "style": "grpAdv",\n' + \
                '  "left": "4em",\n' + \
                '  "top": "2em",\n' + \
                '  "width": "56em",\n' + \
                '  "height": "25em",\n' + \
                '  "visible": "false",\n' + \
                '  "children": [\n'

            children_data = []
            wordCount = 0
            for index, row in df.iterrows():
                if index == 0:
                    continue  # Skip the first row
                if row['type'] == 'vid':
                    child_str = '    {\n'
                    child_str += '      "type":"grp",'
                    child_str += '      "id": "videoGrp01",'
                    child_str += '      "style": "sdwVideo",'
                    child_str += '      "content": "",'
                    child_str += '      "track": "",'
                    child_str += '      "top": "1em",'
                    child_str += '      "left": "1em",'
                    child_str += '      "width": "20em",'
                    child_str += '      "height": "23em"'
                    child_str += '    }'
                    children_data.append(child_str)
                else:
                    child_str = '    {\n'
                    keys = list(row.keys())
                    valid_keys = [key for key in keys if pd.notna(row[key]) and row[key] != ""]
                    specialChar = False
                    for i, key in enumerate(valid_keys):
                        value = row[key]
                        
                        if key == 'style' and value == 'vector':
                            specialChar = True
                            # Add handling for vector styles
                            child_str += (
                                '      "type": "img",\n'
                                '      "style": "specialChar",\n'
                                '      "left": "82em",\n'
                                '      "top": "58em",\n'
                                '      "height": "9em",\n'
                                '      "width": "auto",\n'
                                '      "content": "' + row.get("content", "").replace('"', '\\"') + '"\n'
                            )
                        else:
                            if (specialChar==False):
                                child_str += '      "' + key + '": "' + str(value).replace('"', '\\"') + '"'

                            if i < len(valid_keys) - 1:
                                child_str += ',\n'
                            elif 'style' in row and row['style'] == 'words':
                                child_str += ',\n'
                            else:
                                child_str += '\n'
                    
                    if 'style' in row and row['style'] == 'words':
                        wordCount += 1
                        child_str += (
                            '      "children": [\n'
                            '        {\n'
                            '          "type": "icn",\n'
                            '          "role": "image",\n'
                            '          "content": "' + row.get("icn", "").replace('"', '\\"') + '",\n'
                            '          "left": "-4em",\n'
                            '          "top": "-1em",\n'
                            '          "height": "8em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "btn",\n'
                            '          "id": "",\n'
                            '          "style": "wAudPlay",\n'
                            '          "role": "button",\n'
                            '          "name": "Play Audio",\n'
                            '          "alt": "Play Audio",\n'
                            '          "action": "playAudio",\n'
                            '          "target": "sample.wav",\n'
                            '          "right": "-4em",\n'
                            '          "top": "-1em",\n'
                            '          "height": "8em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "aud",\n'
                            '          "id": "",\n'
                            '          "style": "wAudPlayer",\n'
                            '          "role": "media",\n'
                            '          "action": "",\n'
                            '          "content": "'+row.get("content", "")+'",\n'
                            '          "right": "0em",\n'
                            '          "top": "0em",\n'
                            '          "height": "2em",\n'
                            '          "width": "8em"\n'
                            '        },\n'
                            '        {\n'
                            '          "type": "txt",\n'
                            '          "style": "vocab",\n'
                            '          "content": "' + row.get("text", "").replace('"', '\\"') + '",\n'
                            '          "left": "-.0em",\n'
                            '          "top": ".25em"\n'
                            '        }\n'
                            '      ]\n'
                        )

                    child_str += '    }'
                    children_data.append(child_str)
            
            group += ',\n'.join(children_data) + '\n'
            group += '  ]\n'
            group += '}\n'

            all_data.append(group)

        # Basic Constructor
        elif input_file in ['datacc.xlsx', 'datavv.xlsx']:
            # Handle the basic construction logic here
            row1 = df.iloc[0]
            row2 = df.iloc[1]    
            row3 = df.iloc[2]
            vector = False
            if row3.get("style","") == 'soundwallSub vector':
                vector = True
            if vector:
                all_data.append('{\n'
                    '  "type": "icn",\n'
                    '  "role": "button",\n'
                    '  "alt": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                    '  "id": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                    '  "content": "' + row1.get("content", "").replace('"', '\\"') + '",\n'
                    '  "left": "' + row1.get("left", "").replace('"', '\\"') + '",\n'
                    '  "top": "' + row1.get("top", "").replace('"', '\\"') + '",\n'
                    '  "action": "toggleMulti",\n'
                    '  "target": "grp' + sheet_suffix + '",\n'
                    '  "children": [\n'
                    '                   {'
                    '                       "type": "btn",'
                    '                       "id": "' + str(row2.get('id', 'default_id')) + '",'
                    '                       "alt": "' + str(row2.get('alt', 'default_id')) + '",'
                    '                       "role": "button",\n'
                    '                       "name": "' + str(row1.get('name', 'default_id')) + '",'
                    '                       "style": "hotspot",'
                    '                       "content": "",'
                    '                       "left": "0",'
                    '                       "top": "0em",'
                    '                       "height": "9.5em",'
                    '                       "width": "9.5em"'
                    '                   },'
                    '                   {'
                    '                        "type": "grp",'
                    '                        "id": "' + str(row3.get('id', 'default_id')) + '",'
                    '                        "content": "' + str(row3.get('content', 'default_content')) + '",'
                    '                        "alt": "' + str(row3.get('alt', 'default_id')) + '",'
                    '                        "left": ".4em",'
                    '                        "top": "11em",'
                    '                        "width": "9em",'
                    '                        "height": "12em",'
                    '                        "visible": "false",'
                    '                        "children": ['
                    '                            {'
                    '                            "type": "img",'
                    '                            "style": "altFont",'
                    '                            "left": "6em",'
                    '                            "content": "' + str(row3.get('text', 'default_text')) + '",'
                    '                            "left": ".5em",'
                    '                            "top": ".25em",'
                    '                            "height": "3em",'
                    '                            "width": "7em"'
                    '                            },'
                    '                            {'
                    '                                "type": "img",'
                    '                                "style": "photo",'
                    '                                "alt": "' + str(row3.get('alt', 'default_alt')) + '",'
                    '                                "content": "' + str(row3.get('content', 'default_content')) + '",'
                    '                                "left": ".5em",'
                    '                                "top": "3.5em",'
                    '                                "width": "7.5em",'
                    '                                "height": "auto"'
                    '                            }'
                    '                        ]\n'
                    '                    }\n'
                    '                 ]\n'
                    
                    '}'
                )




            if vector == False:

                all_data.append('{\n'
                    '  "type": "icn",\n'
                    '  "alt": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                    '  "id": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                    '  "content": "' + row1.get("content", "").replace('"', '\\"') + '",\n'
                    '  "left": "' + row1.get("left", "").replace('"', '\\"') + '",\n'
                    '  "top": "' + row1.get("top", "").replace('"', '\\"') + '",\n'
                    '  "action": "toggleMulti",\n'
                    '  "target": "grp' + sheet_suffix + '",\n'
                    '  "children": [\n'
                    '                   {'
                    '                       "type": "btn",'
                    '                       "id": "' + str(row2.get('id', 'default_id')) + '",'
                    '                       "alt": "' + str(row2.get('alt', 'default_id')) + '",'
                    '                       "role": "button",\n'
                    '                       "name": "' + str(row1.get('name', 'default_id')) + '",'
                    '                       "style": "hotspot",'
                    '                       "content": "",'
                    '                       "left": "0",'
                    '                       "top": "0em",'
                    '                       "height": "9.5em",'
                    '                       "width": "9.5em"'
                    '                   },'
                    '                   {'
                    '                        "type": "grp",'
                    '                        "id": "' + str(row3.get('id', 'default_id')) + '",'
                    '                        "content": "' + str(row3.get('content', 'default_content')) + '",'
                    '                        "alt": "' + str(row3.get('alt', 'default_id')) + '",'
                    '                        "left": ".4em",'
                    '                        "top": "11em",'
                    '                        "width": "9em",'
                    '                        "height": "12em",'
                    '                        "visible": "false",'
                    '                        "children": ['
                    '                            {'
                    '                                "type": "txt",'
                    '                                "content": "' + str(row3.get('text', 'default_text')) + '",'
                    '                                "left": "-.15em",'
                    '                                "top": "-.15em"'
                    '                            },'
                    '                            {'
                    '                                "type": "img",'
                    '                                "style": "photo",'
                    '                                "alt": "' + str(row3.get('alt', 'default_alt')) + '",'
                    '                                "content": "' + str(row3.get('content', 'default_content')) + '",'
                    '                                "left": ".5em",'
                    '                                "top": "3.5em",'
                    '                                "width": "7.5em",'
                    '                                "height": "auto"'
                    '                            }'
                    '                        ]\n'
                    '                    }\n'
                    '                 ]\n'
                    
                    '}'
                )

    # Write to JSON file with utf-8 encoding
    with open(output_file, 'w', encoding='utf-8') as json_file:
        json_file.write(',\n'.join(all_data))

# Specify the input and output file paths for each Excel file
file_mappings = {
    "datavv.xlsx": "output1.json",
    "datavv2.xlsx": "output2.json",
    "datacc.xlsx": "output3.json",
    "datacc2.xlsx": "output4.json"
}

# Process each file
for input_file, output_file in file_mappings.items():
    excel_to_json_string(input_file, output_file)


Now we will insert the json snippets into the framework file.

In [2]:
import os

def replace_placeholder_with_json(shell_file, output_files, final_output):
    # Read the shell file content
    with open(shell_file, 'r', encoding='utf-8') as shell:
        shell_content = shell.read()

    # Replace placeholders with the corresponding output file content
    for output_file in output_files:
        placeholder = f'/*ADD {os.path.basename(output_file)} here*/'
        with open(output_file, 'r', encoding='utf-8') as output:
            output_content = output.read()
        shell_content = shell_content.replace(placeholder, output_content)

    # Write the final content to the new file
    with open(final_output, 'w', encoding='utf-8') as final:
        final.write(shell_content)

# Paths to the files
shell_file = '../data/soundwallShell.json'
output_files = [
    'output1.json',
    'output2.json',
    'output3.json',
    'output4.json'
]
final_output = '../data/presoundwall.json'

# Perform the replacement
replace_placeholder_with_json(shell_file, output_files, final_output)

print(f'Final JSON file created at {final_output}')


Final JSON file created at ../data/presoundwall.json


fix triple commas

In [3]:
import re

def fix_broken_json(input_file, output_file):
    # Read the entire content of the broken JSON file
    with open(input_file, 'r', encoding='utf-8') as file:
        content = file.read()

    # Fix common issues
    content = re.sub(r',\s*,\s*,', ',', content)  # Replace triple commas with a single comma
    content = re.sub(r',\s*,', ',', content)      # Replace double commas with a single comma
    content = re.sub(r'\[\s*,', '[', content)     # Remove leading commas in arrays
    content = re.sub(r',\s*\]', ']', content)     # Remove trailing commas in arrays
    content = re.sub(r',\s*}', '}', content)      # Remove trailing commas before closing braces
    content = re.sub(r'{\s*,', '{', content)      # Remove leading commas after opening braces

    # Write the fixed content back to a new file
    with open(output_file, 'w', encoding='utf-8') as file:
        file.write(content)

    print(f"Fixed JSON file saved to {output_file}")

# Specify the input and output file paths
input_file = '../data/presoundwall.json'  # Update with the correct path
output_file = '../data/soundwall.json'  # Desired output file path

# Run the fix
fix_broken_json(input_file, output_file)


Fixed JSON file saved to ../data/soundwall.json
